In [1]:
import logging
import tomobase
tomobase.bootstrap(jupyter_enabled=True, qt_enabled=True)

from tomobase import phantoms, procedures, data_classes, logger, proxy, GPUContext
from tomobase import jupyter

logger.setLevel(logging.DEBUG)
proxy.set_context(GPUContext.CUPY)
jupyter.display_log() 

type tester <class 'type'> <class 'type'> <class 'type'>
called qt


In [2]:
import os
import imageio
import numpy as np
import pathlib
from tomobase.core.data_classes.images import Sinogram
_base_dir = r"\\ematbyname\emattecnai\Liam\23042026\P2"

n, s = len(os.listdir(_base_dir))-1, 2
x, y = None, None
angles = np.zeros(n)
data_dim = 0
isdatafile = False
contains_none = 0
for i, folder in enumerate(os.listdir(_base_dir)):
    folder_path = os.path.join(_base_dir, folder)
    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            if 'VDF_20_90' in file and file.endswith('.tiff'):
                path = os.path.join(folder_path, file)
                data = imageio.imread(path)
                data_dim=1
                isdatafile = True
            elif 'VDF_235_512' in file and file.endswith('.tiff'):
                angles[i] = float(pathlib.Path(file).stem.split('e')[-1])
                path = os.path.join(folder_path, file)
                data = imageio.imread(path)
                isdatafile = True
                data_dim=0
            
            if isdatafile:
                if i == 0 and x is None and y is None:
                    x, y = data.shape
                    array = np.zeros((n, s, y, x)) 
                if data_dim == 0:
                    array[i, 0, ...] = data
                elif data_dim == 1:
                    array[i, 1, ...] = data
                isdatafile = False
                
print(array.shape, angles)
sinogram = Sinogram('VDF_20', array, angles)
sinogram.signal_labels = ["HAADF", "DF"]
print(sinogram.xr.sizes["signals"])
sinogram.interactive.info()

C:\Users\tcrai\AppData\Local\Temp\ipykernel_24568\3782351650.py:20: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  data = imageio.imread(path)
C:\Users\tcrai\AppData\Local\Temp\ipykernel_24568\3782351650.py:26: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  data = imageio.imread(path)


(49, 2, 1024, 1024) [ 70.  67.  64.  61.  58.  55.  52.  49.  46.  43.  40.  37.  34.  31.
  28.  25.  22.  19.  16.  13.  10.   7.   4.   1.  -2.  -5.  -8. -11.
 -14. -17. -20. -23. -26. -29. -32. -35. -38. -41. -41. -44. -47. -50.
 -53. -56. -59. -62. -65. -68. -71.]
2


InfoWidget(children=(ImageSliceWidget(children=(ToggleButtons(description='View:', options=(('ny', ('n', 'y'))…

In [ ]:
from tomobase.core.data_classes.images import Sinogram

import hyperspy.api as hs
path = r"\\ematbyname\emattitan3\Aina\2026.04.10\2026.04.10 EELS\20260410092201_EELS_SET_NP1-00.hspy", 
             


signals = hs.load(path)

# Inspect loaded signals
for i, sig in enumerate(signals):
    print(i, sig)

# Pick one EELS spectrum image
s = signals[1]   # low-loss? energy_offset_index=0
# s = signals[1] # core-loss? energy_offset_index=1

print(s)
print(s.axes_manager)

# Plot full spectrum image
s.plot()

# Extract image at a specific energy
energy = 2.4 # eV, change this

img = s.isig[energy]
img.plot()

tolerance = 0.05  # eV, change this
# Better: integrate over an energy window
e_min = energy - tolerance  # eV, change this
e_max = energy + tolerance  # eV, change this

img_window = s.isig[e_min:e_max].sum(axis=-1)
print("hello", img_window)
img_window.plot()

# Optional: compare with HAADF
haadf = signals[2]
haadf.plot()

In [3]:
from tomobase import procedures, phantoms, progress, registers
import scipy

#print(scipy.ndimage.center_of_mass(arr))

sinogram2 = procedures.align_sinogram_center_of_mass(sinogram, inplace=False)
sinogram2 = procedures.align_sinogram_xcorr(sinogram2)


ProgressBarWidget(children=(Label(value='Aligning sinogram with center of mass'), HBox(children=(IntProgress(v…

ProgressBarWidget(children=(Label(value='Aligning signals to COM'), HBox(children=(IntProgress(value=0, max=98…

ProgressBarWidget(children=(Label(value='Calculating shifts with cross-correlation'), HBox(children=(IntProgre…

ProgressBarWidget(children=(Label(value='Aligning sinogram with cross-correlation'), HBox(children=(IntProgres…

In [ ]:
sinogram3 = procedures.bin(sinogram2, inplace=False)
#sinogram3.interactive.info()



In [8]:
import os

from tomobase.domain import io

vol = procedures.reconstruct_mlem(sinogram4, iterations=15)
file_name = ['HAADF', 'DF']
path =  r'\\ematbyname\emat\TimC'
folder = vol.name+'3'
#os.mkdir(os.path.join(path, folder))
for i, item in enumerate(vol.split('signals')):
    path_name = os.path.join(path, folder, f"{file_name[i]}_recon.rec")
    vol_color = item
    
    
io.rec._write_rec(vol_color, path_name)
vol_color.interactive.info()

ProgressBarWidget(children=(Label(value='Iterative Back Projection'), HBox(children=(IntProgress(value=0, max=…

ProgressBarWidget(children=(Label(value='Slice Wise Back Projection'), HBox(children=(IntProgress(value=0, max…

ProgressBarWidget(children=(Label(value='Slice Wise Back Projection'), HBox(children=(IntProgress(value=0, max…

ProgressBarWidget(children=(Label(value='Slice Wise Back Projection'), HBox(children=(IntProgress(value=0, max…

ProgressBarWidget(children=(Label(value='Slice Wise Back Projection'), HBox(children=(IntProgress(value=0, max…

...

ProgressBarWidget(children=(Label(value='Slice Wise Back Projection'), HBox(children=(IntProgress(value=0, max…

ProgressBarWidget(children=(Label(value='Slice Wise Back Projection'), HBox(children=(IntProgress(value=0, max…

ProgressBarWidget(children=(Label(value='Slice Wise Back Projection'), HBox(children=(IntProgress(value=0, max…

ProgressBarWidget(children=(Label(value='Slice Wise Back Projection'), HBox(children=(IntProgress(value=0, max…

InfoWidget(children=(ImageSliceWidget(children=(ToggleButtons(description='View:', options=(('xy', ('x', 'y'))…

In [15]:
vol_color = procedures.bin(vol_color)
vol_color.interactive.info(display_type="volume")

InfoWidget(children=(ImageVolumeWidget(children=(ToggleButtons(description='View:', options=(('xyz', ('x', 'y'…

In [ ]:
sinogram4 = procedures.align_tilt_axis_shift(sinogram3, inplace=False)
sinogram4 = procedures.align_tilt_axis_rotation(sinogram4)

tomobase.jupyter.SliceGrid((sinogram4, sinogram3), columns=2)

ProgressBarWidget(children=(Label(value='Calculating tilt axis shift'), HBox(children=(IntProgress(value=0, ma…

ProgressBarWidget(children=(Label(value='Iterative Back Projection'), HBox(children=(IntProgress(value=0, max=…

ProgressBarWidget(children=(Label(value='Slice Wise Back Projection'), HBox(children=(IntProgress(value=0, max…

ProgressBarWidget(children=(Label(value='Slice Wise Back Projection'), HBox(children=(IntProgress(value=0, max…

ProgressBarWidget(children=(Label(value='Slice Wise Back Projection'), HBox(children=(IntProgress(value=0, max…

...

ProgressBarWidget(children=(Label(value='Slice Wise Back Projection'), HBox(children=(IntProgress(value=0, max…

ProgressBarWidget(children=(Label(value='Slice Wise Back Projection'), HBox(children=(IntProgress(value=0, max…

ProgressBarWidget(children=(Label(value='Forward projecting'), HBox(children=(IntProgress(value=0, max=2), Lab…

ProgressBarWidget(children=(Label(value='Applying tilt axis shift'), HBox(children=(IntProgress(value=0, max=2…

ValueError: too many values to unpack (expected 2)

In [ ]:
import os
import sys

for p in sys.path:
    if "mesa" in p.lower():
        print(p)

print(os.environ.get("PATH"))

In [ ]:
print(sinogram3.data.sizes)
sinogram2.interactive.info()